[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-02-information-extraction.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 2 — Information Extraction with Regular Expressions

**MSIS · AME 5053 · Week 2 · 3 hours**

Last week you set up Colab and refreshed Python. Today you do your first real NLP
job: **information extraction** — pulling structured facts out of messy, unstructured
text.

You will work on one realistic mess all lab: a pile of bank SMS messages. By the end
you will have turned lines like

> `Rs.2,500.00 debited from a/c XX3481 on 05-04-2026 to SWIGGY. Avl Bal Rs.18,340.55`

into tidy records a program can use:

> `{amount: 2500.00, account: XX3481, date: 05-04-2026, merchant: SWIGGY}`

And you will see exactly where regular expressions stop being the right tool — which
is the whole reason the rest of this course exists.

**By the end of this lab you will be able to:**

1. Find every match of a pattern, with `regex.findall` and `regex.finditer`
2. Pull out the piece you want using **capturing groups**, named for readability
3. Avoid the **greedy trap** that catches everyone once
4. Handle Kannada and other Indian-language text correctly — where the built-in
   `re` silently corrupts your results
5. Say clearly what regex *cannot* do, and what comes next

Run every cell, in order.

---
## Part 0 — Setup

You did the full Colab tour last week, so this is short. But **Colab wipes the machine
between sessions**, so you still install every time. Run these two cells.

In [ ]:
# Runs at the top of every lab. Colab forgets everything between sessions.
!pip install -q regex
print("Done.")

In [ ]:
# Check the one library this lab needs.
import sys
try:
    import regex
    print("regex", regex.__version__, "— ready")
except ImportError:
    print("MISSING. Re-run the install cell above.")
print("Python", sys.version.split()[0])

> **Save your own copy now:** File → Save a copy in Drive. Work in that copy, or you
> lose your work when the tab closes. (Yes, every week.)

We use **`regex`**, not Python's built-in `re`. They look identical, and for plain
English they behave identically. But `re` breaks on Kannada and other Indian scripts —
you will see it fail with your own eyes in Part 7. Get in the habit now: `import regex`.

---
## Part 1 — What is information extraction?

Most text in the world is **unstructured**: written for humans, with no fixed fields. A
bank SMS is a sentence, not a spreadsheet row. **Information extraction** is the job of
reading that text and pulling out the specific facts you care about — the amount, the
date, the account — and putting them into a structure a program can use.

Regular expressions are the oldest tool for this, and still the right one when the thing
you want has a **predictable shape**: a date looks like a date, a rupee amount looks like
a rupee amount. Today's whole job is: describe those shapes precisely, then let the
computer find every one.

Here is our mess for the day. Run it.

In [ ]:
messages = [
    "Rs.2,500.00 debited from a/c XX3481 on 05-04-2026 to SWIGGY. Avl Bal Rs.18,340.55",
    "INR 799 spent on card XX9012 at AMAZON on 06/04/2026. Not you? Call 18001234567",
    "Your a/c XX3481 credited with Rs.45,000.00 on 01-04-2026 SALARY. Avl Bal Rs.63,340.55",
    "Rs.150 debited from a/c XX3481 on 07-04-2026 to AUTO RICKSHAW UPI. Avl Bal Rs.18,190.55",
    "Payment of Rs.1,25,000 to LIC received on 28-03-2026. Ref 9876543210",
    "OTP 458213 for txn of Rs.9,999 at FLIPKART. Valid 10 min. Do not share.",
]

print(len(messages), "messages")
for m in messages:
    print("-", m)

# Verified output:
#   6 messages
#   - Rs.2,500.00 debited from a/c XX3481 on 05-04-2026 to SWIGGY. Avl Bal Rs.18,340.55
#   - INR 799 spent on card XX9012 at AMAZON on 06/04/2026. Not you? Call 18001234567
#   - Your a/c XX3481 credited with Rs.45,000.00 on 01-04-2026 SALARY. Avl Bal Rs.63,340.55
#   - Rs.150 debited from a/c XX3481 on 07-04-2026 to AUTO RICKSHAW UPI. Avl Bal Rs.18,190.55
#   - Payment of Rs.1,25,000 to LIC received on 28-03-2026. Ref 9876543210
#   - OTP 458213 for txn of Rs.9,999 at FLIPKART. Valid 10 min. Do not share.

---
## Part 2 — Find every match

Three functions do almost everything in this lab. You met them in sessions 1 and 2;
here they are on our data.

- **`regex.findall(pattern, text)`** — a list of every match.
- **`regex.search(pattern, text)`** — the first match only, as a match object (or `None`).
- **`regex.finditer(pattern, text)`** — every match as an object, so you can ask *where*
  it is.

Start with the account numbers. They all look the same: `XX` then four digits. In a
pattern, `\d` means "a digit" and `{4}` means "exactly four of them".

In [ ]:
import regex

text = "Rs.2,500.00 debited from a/c XX3481 on 05-04-2026 to SWIGGY. Avl Bal Rs.18,340.55"

print(regex.findall(r"XX\d{4}", text))     # every account-like token
print(regex.search(r"XX\d{4}", text))      # the first one, as a match object

m = regex.search(r"XX\d{4}", text)
print("matched text:", m.group())          # what it found
print("position:", m.start(), "to", m.end())

# Verified output:
#   ['XX3481']
#   <regex.Match object; span=(29, 35), match='XX3481'>
#   matched text: XX3481
#   position: 29 to 35

Now run the same account pattern across **all** the messages. This is the real shape of
an extraction job: one pattern, a whole pile of text.

In [ ]:
import regex

accounts = []
for m in messages:
    accounts += regex.findall(r"XX\d{4}", m)

print("all account mentions:", accounts)

from collections import Counter
print("how often each appears:", Counter(accounts))

# Verified output:
#   all account mentions: ['XX3481', 'XX9012', 'XX3481', 'XX3481']
#   how often each appears: Counter({'XX3481': 3, 'XX9012': 1})

### Exercise 1

Every message has a rupee amount, but written several ways: `Rs.2,500.00`, `INR 799`,
`Rs.150`, `Rs.1,25,000`. For now, just find the **numbers that follow `Rs.`** — ignore
the `INR` ones, we handle those next part.

Use `\d` for a digit, `,` for a literal comma, `.` you will need to escape as `\.`
because a bare `.` means "any character".

Expected: the `Rs.` amounts, e.g. `['2,500.00', '18,340.55']` from the first message.

In [ ]:
# YOUR CODE HERE
# Hint: match Rs\. then one or more of (digit, comma, dot).
# Hint: [\d,.]+  means "one or more characters that are a digit, comma, or dot".

---
## Part 3 — One pattern for every amount

The amounts come two ways: `Rs.2,500.00` and `INR 799`. You could write two patterns.
Better: write one that accepts **either** prefix. The `|` symbol means "or", and you
group the alternatives with `( ... )`.

- `Rs\.?` — the letters `Rs`, then an optional dot (`?` means "zero or one of the thing
  before it").
- `INR` — the letters `INR`.
- `(?:Rs\.?|INR)` — either of those. The `?:` says "group these, but don't capture them
  as a result" — we only want the number, not the prefix.

In [ ]:
import regex

# (?:Rs\.?|INR)  = the prefix, either form, not captured
# \s?            = an optional space (INR 799 has one, Rs.150 does not)
# ([\d,]+(?:\.\d{2})?) = the amount: digits/commas, then an OPTIONAL .00 part
money = r"(?:Rs\.?|INR)\s?([\d,]+(?:\.\d{2})?)"

for m in messages:
    print(regex.findall(money, m))

# Verified output:
#   ['2,500.00', '18,340.55']
#   ['799']
#   ['45,000.00', '63,340.55']
#   ['150', '18,190.55']
#   ['1,25,000']
#   ['9,999']

Look at what came back. Every amount, in both formats — and notice it also caught the
**balance** (`Avl Bal Rs.18,340.55`), because that is a rupee amount too. The pattern
did exactly what you said, not what you meant. That is regex in one sentence: it is
literal, always.

We will separate "amount" from "balance" properly in Part 5, using their context.

### Turning the text into a number

`'2,500.00'` is still a string. To do arithmetic you strip the commas and call `float`.

In [ ]:
import regex

money = r"(?:Rs\.?|INR)\s?([\d,]+(?:\.\d{2})?)"

def to_number(s):
    return float(s.replace(",", ""))

amounts = []
for m in messages:
    for hit in regex.findall(money, m):
        amounts.append(to_number(hit))

print("as numbers:", amounts)
print("largest:", max(amounts))

# Verified output:
#   as numbers: [2500.0, 18340.55, 799.0, 45000.0, 63340.55, 150.0, 18190.55, 125000.0, 9999.0]
#   largest: 125000.0

### Exercise 2

Notice `Rs.1,25,000` — that is the **Indian grouping**: one lakh twenty-five thousand,
written `1,25,000`, not `125,000`. Our pattern handled it fine, because `[\d,]+` does not
care *where* the commas fall.

Your turn: from the whole pile, extract every amount as a number, and print the **total
debited** — but only from messages containing the word `debited`. (Two of them do.)

In [ ]:
# YOUR CODE HERE
# Hint: loop the messages; skip any without "debited" in it.
# Hint: within a debited message, the FIRST amount is the transaction amount.

---
## Part 4 — Dates, and choosing a separator

The dates come two ways: `05-04-2026` (dashes) and `06/04/2026` (slashes). One pattern
handles both if you let the separator be "a dash or a slash": `[-/]`.

A character class `[ ... ]` means "any one of these characters". So `[-/]` is "a dash or
a slash". (A dash is safe at the very start or end of a class; elsewhere it means a
range, like `[0-9]`.)

In [ ]:
import regex

# \d{2}  two digits, then a separator, then two digits, then sep, then four digits
date = r"\d{2}[-/]\d{2}[-/]\d{4}"

for m in messages:
    print(regex.findall(date, m))

# Verified output:
#   ['05-04-2026']
#   ['06/04/2026']
#   ['01-04-2026']
#   ['07-04-2026']
#   ['28-03-2026']
#   []

Five messages have a date; one (the OTP message) does not, so it returns an empty list.
That is normal — extraction patterns miss cleanly rather than crash.

### Exercise 3

Build one loop that produces a small table: for each message that has both, print its
**date** and its **account**. Skip messages missing either.

Expected: four rows (the OTP and LIC messages are missing one field each).

In [ ]:
# YOUR CODE HERE
# Hint: use regex.search (not findall) to get the first of each, and check for None.

---
## Part 5 — From a line of text to a record

This is the real goal of information extraction: turn one message into a **structured
record** — a dictionary with named fields. Doing it one field at a time works, but there
is a cleaner way that reads the whole message shape at once: **named groups**.

`(?P<amount>...)` is a capturing group with a name. After a match you pull each field out
by name with `m.group("amount")` — or get them all at once with `m.groupdict()`.

Here is the pattern for a debit message, written across several lines so you can read it.

In [ ]:
import regex

debit = regex.compile(r"""
    Rs\.(?P<amount>[\d,]+(?:\.\d{2})?)   # the transaction amount (paise optional)
    \s+debited\s+from\s+a/c\s+
    (?P<account>XX\d{4})                 # the account
    \s+on\s+
    (?P<date>\d{2}-\d{2}-\d{4})          # the date
    \s+to\s+
    (?P<merchant>[A-Z ]+?)               # the merchant (lazy — see Part 6)
    \.\s+Avl\s+Bal\s+Rs\.
    (?P<balance>[\d,]+\.\d{2})           # the closing balance
""", regex.VERBOSE)

m = debit.search(messages[0])
print(m.groupdict())

# Verified output:
#   {'amount': '2,500.00', 'account': 'XX3481', 'date': '05-04-2026', 'merchant': 'SWIGGY', 'balance': '18,340.55'}

`regex.VERBOSE` (the triple-quoted pattern) lets you spread a pattern over many lines
with comments. For anything longer than a few tokens, write it this way — a wall of
`\d{2}[-/]\d{2}` is unreadable and unmaintainable.

`m.groupdict()` gave you a ready-made record. Now run it over every message and collect
the debits into a clean table.

In [ ]:
import regex

debit = regex.compile(r"""
    Rs\.(?P<amount>[\d,]+(?:\.\d{2})?)
    \s+debited\s+from\s+a/c\s+
    (?P<account>XX\d{4})
    \s+on\s+(?P<date>\d{2}-\d{2}-\d{4})
    \s+to\s+(?P<merchant>[A-Z ]+?)
    \.\s+Avl\s+Bal\s+Rs\.(?P<balance>[\d,]+\.\d{2})
""", regex.VERBOSE)

records = []
for m in messages:
    hit = debit.search(m)
    if hit:
        records.append(hit.groupdict())

for r in records:
    print(r)

print()
print("extracted", len(records), "debit records")

# Verified output:
#   {'amount': '2,500.00', 'account': 'XX3481', 'date': '05-04-2026', 'merchant': 'SWIGGY', 'balance': '18,340.55'}
#   {'amount': '150', 'account': 'XX3481', 'date': '07-04-2026', 'merchant': 'AUTO RICKSHAW UPI', 'balance': '18,190.55'}
#   
#   extracted 2 debit records

---
## Part 6 — The greedy trap

In Part 5 the merchant group was `[A-Z ]+?` — with a `?` on the end. That `?` is doing
important work. Here is what happens without it.

By default `+` and `*` are **greedy**: they grab as much as they possibly can while still
letting the rest of the pattern match. Watch a greedy merchant group swallow half the
message.

In [ ]:
import regex

text = "Rs.2,500.00 debited from a/c XX3481 on 05-04-2026 to SWIGGY. Avl Bal Rs.18,340.55"

# GREEDY: .+ grabs as much as it can, backing off only as much as it must
greedy = regex.search(r"to (?P<merchant>.+)\.", text)
print("greedy :", repr(greedy.group("merchant")))

# LAZY: .+? grabs as LITTLE as it can
lazy = regex.search(r"to (?P<merchant>.+?)\.", text)
print("lazy   :", repr(lazy.group("merchant")))

# Verified output:
#   greedy : 'SWIGGY. Avl Bal Rs.18,340'
#   lazy   : 'SWIGGY'

The greedy `.+` ran all the way to the **last** dot in the line, capturing
`SWIGGY. Avl Bal Rs.18,340` — everything up to the final `.` — when all you wanted was
`SWIGGY`. The lazy `.+?` stopped at the **first** dot instead, and got it right.

**This is the single most common regex bug**, and now you have written it once on
purpose. The fixes, in order of preference:

1. **Be specific instead of using `.`** — our Part 5 pattern used `[A-Z ]+?`, which
   physically cannot cross into the digits of the balance. Specific beats lazy.
2. **Use lazy `+?` / `*?`** when you genuinely need "any character, but stop early".

### Exercise 4

The greedy pattern below is meant to grab just the merchant `AMAZON`, but it grabs too
much. Fix it two ways: once by making it lazy, once by making the character class
specific. Both should print `AMAZON`.

In [ ]:
text = "INR 799 spent on card XX9012 at AMAZON on 06/04/2026. Not you? Call 18001234567"

# This grabs too much — fix it.
# import regex
# print(regex.search(r"at (.+) on", text).group(1))

# YOUR CODE HERE — two fixes

---
## Part 7 — Why we never use the built-in `re`

Everything so far would work with Python's built-in `re` too. This part is why we do not
use it.

Banks send SMS in Indian languages. Here is the same kind of message with a Kannada
merchant name. We want to pull out the words. `\w+` means "one or more word
characters" — the obvious tool.

In [ ]:
import re        # the BUILT-IN one, for contrast
import regex     # the one we use

sms = "ಬೆಂಗಳೂರು ಮಾರುಕಟ್ಟೆ Rs.500 ಡೆಬಿಟ್"   # a Kannada bank SMS

print("built-in re :", re.findall(r"\w+", sms))
print("regex       :", regex.findall(r"\w+", sms))

# Verified output:
#   built-in re : ['ಬ', 'ಗಳ', 'ರ', 'ಮ', 'ರ', 'ಕಟ', 'ಟ', 'Rs', '500', 'ಡ', 'ಬ', 'ಟ']
#   regex       : ['ಬೆಂಗಳೂರು', 'ಮಾರುಕಟ್ಟೆ', 'Rs', '500', 'ಡೆಬಿಟ್']

Look carefully at the two lists.

The word `ಮಾರುಕಟ್ಟೆ` (*maarukatte*, "market") is **one** word. The built-in `re` splits
it into pieces, because Kannada joins consonants with a mark called the **virama**
(the little stroke in `ಟ್ಟೆ`), and `re`'s idea of `\w` does not count that mark as part
of a word. So it cuts the word at the joint.

`regex` knows about the virama and keeps the word whole.

Count them, so the damage is undeniable.

In [ ]:
import re, regex

sms = "ಬೆಂಗಳೂರು ಮಾರುಕಟ್ಟೆ Rs.500 ಡೆಬಿಟ್"

print("re    found", len(re.findall(r"\w+", sms)), "tokens")
print("regex found", len(regex.findall(r"\w+", sms)), "tokens")
print()
print("The right answer is 5: three Kannada words, plus 'Rs' and '500'.")
print("regex gets 5. re shatters the three Kannada words into pieces.")

# Verified output:
#   re    found 12 tokens
#   regex found 5 tokens
#   
#   The right answer is 5: three Kannada words, plus 'Rs' and '500'.
#   regex gets 5. re shatters the three Kannada words into pieces.

This is not a rare edge case in Karnataka — it is every second line of real text. And it
fails **silently**: no error, just wrong counts feeding everything downstream. Every
word count, every frequency, every model built on top inherits the damage and nobody
sees where it started.

**So: `import regex`, never `import re`.** It is a drop-in replacement — same functions,
same patterns — that gets Indian scripts right. This one habit is worth the whole lab.

---
## Part 8 — Where regex stops being enough

You have done a lot with patterns today. Now the honest part: two things regex genuinely
cannot do, no matter how clever the pattern. Both are the reason the rest of this course
exists.

### 1. It reads the shape, not the meaning

Look at this date again.

In [ ]:
import regex

text = "INR 799 spent on card XX9012 at AMAZON on 06/04/2026."
print("extracted date string:", regex.search(r"\d{2}/\d{2}/\d{4}", text).group())

# Verified output:
#   extracted date string: 06/04/2026

Regex pulled out `06/04/2026` perfectly. But **what date is it?** In India that is the
6th of April. Written by an American system it would be the 4th of June. The *string* is
unambiguous; the *meaning* is not — and no pattern can decide it, because the answer is
not in the shape of the text. It is in knowledge the text does not contain.

### 2. Some fields have no shape to match

Look back at the merchants we extracted: `SWIGGY`, `AMAZON`, `FLIPKART`, `LIC`, and
`AUTO RICKSHAW UPI`. Our tidy `[A-Z]+` pattern assumed a merchant is one block of capital
letters. Reality disagrees.

In [ ]:
import regex

# Our "merchant is one all-caps word" assumption, on the messy message:
text = "Rs.150 debited from a/c XX3481 on 07-04-2026 to AUTO RICKSHAW UPI. Avl Bal Rs.18,190.55"
print("[A-Z]+ grabs:", regex.search(r"to ([A-Z]+)", text).group(1))
print("...but the real merchant is 'AUTO RICKSHAW', and 'UPI' is just noise.")

# Verified output:
#   [A-Z]+ grabs: AUTO
#   ...but the real merchant is 'AUTO RICKSHAW', and 'UPI' is just noise.

`[A-Z]+` stops at the first space, so it grabs only `AUTO`. Widen it to `[A-Z ]+` and it
grabs `AUTO RICKSHAW UPI` — now including the `UPI` that is not part of the name. There
is **no pattern** that reliably knows where a merchant name starts and ends, because
merchant names have no fixed shape. `SWIGGY` and `AUTO RICKSHAW UPI` and `LIC` share
nothing a regex can grab onto.

**This is the wall.** Deciding "this span of text is a company, that one is a date, that
one is a person" is called **Named Entity Recognition**, and you meet it in session 7. It
does not use hand-written patterns — it **learns** what an entity looks like from
labelled examples, the same shift you saw last week when VADER failed on *nightmare*.

Regex is the right tool when the thing has a shape: account numbers, dates, rupee
amounts — you nailed all of those today, and you will use them for the rest of your
career. It is the wrong tool the moment you need meaning. Knowing which is which is the
skill.

---
## Before you go

Check all of these:

- [ ] **Runtime → Restart and run all** works top to bottom with no errors
- [ ] You can pull a field out of text with a **named group** and `groupdict()`
- [ ] You can explain the **greedy trap** and two ways to fix it
- [ ] You saw `re` break on Kannada and `regex` get it right — and you know why
- [ ] You can name one thing regex **cannot** do, and what comes next
- [ ] You saved your own copy to Drive

**Take-home (optional, 15 min):** the credit message
(`Your a/c XX3481 credited with Rs.45,000.00 on 01-04-2026 SALARY...`) does **not** match
our debit pattern — it is worded differently. Write a second named-group pattern for
credit messages. Notice how each new message shape needs its own rule. Hold that thought:
it is the reason session 7 exists.

If any box is unticked, ask now.

**Next lab:** tokenization, normalization, stemming and stop-word removal — cleaning text
up before you do anything else with it.